## Análisis principal

In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.utils.extmath import randomized_svd
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn import metrics
from sklearn.metrics import pairwise_distances
import seaborn as sns

#### Importar datos

In [2]:
df_aleman = pd.read_excel("Aleman_Principal.xlsx")

In [3]:
df_chino = pd.read_excel("Chino_Principal.xlsx")

In [4]:
df_ruso = pd.read_excel("Ruso_Principal.xlsx")

In [5]:
df_aleman['Idioma'] = 'Alemán'
df_chino['Idioma'] = 'Chino'
df_ruso['Idioma'] = 'Ruso'

#### Preparar datos

In [6]:
datasets = [df_aleman,df_chino,df_ruso]
df = pd.concat(datasets, ignore_index=True)
df = df.fillna(0)

In [7]:
#df['ID1'].value_counts() -> comprobar si hay duplicados

In [8]:
columnas_estrategias = [
    col for col in df.columns if col.startswith("E") and "[SQ" in col
]

Estrategias usadas (a elegir, selección múltiple):
- SQ001: Usar mis conocimientos de otros idiomas (lengua materna u otras lenguas extranjeras)
- SQ002: Analizar la forma de la palabra o sus partes
- SQ003: Sacar conclusiones del contexto inmediato (p.ej. relación con otras palabras en la frase)
- SQ004: Sacar conclusiones del contexto general (p.ej. tema del texto o de la clase)
- SQ005: Fijarme en elementos no verbales (p.ej. imágenes, el formato del texto)
- SQ006: Consultar un diccionario o un traductor online
- SQ007: Usar una aplicación de Inteligencia Artificial (p.ej. Chat GPT)
- SQ008: Preguntar a un/a compañero/a
- SQ009: Preguntar al/a la docente

In [9]:
for col in columnas_estrategias:
    df[col] = (
        df[col]
        .astype(str)
        .str.lower()
        .str.strip()
        .map({"yes": 1, "no": 0})
    )

##### Uso de tipo de estrategias en el grupo 

In [10]:
frecuencias_totales = {}

for col in columnas_estrategias:
    codigo_estrategia = col.split("[")[1].split("]")[0]

    votos = df[col].sum()
    frecuencias_totales[codigo_estrategia] = (
        frecuencias_totales.get(codigo_estrategia, 0) + votos
    )

In [11]:
df_resultados = pd.DataFrame(
    list(frecuencias_totales.items()), columns=["Código", "Frecuencia"]
)
total_elecciones = df_resultados["Frecuencia"].sum().astype(int)
df_resultados["Porcentaje"] = np.round(((
    df_resultados["Frecuencia"] / total_elecciones
) * 100), 2)

In [12]:
df_resultados_gen = df_resultados.replace({'SQ001': 'Otros idiomas', 
                        'SQ002': 'Analizar forma',
                        'SQ003': 'Contexto inmediato',
                      'SQ004': 'Contexto general',
                      'SQ005': 'Elementos no verbales',
                      'SQ006': 'Diccionario/Traductor',
                      'SQ007': 'Aplicación IA',
                      'SQ008': 'Preguntar compañero/a',
                      'SQ009': 'Preguntar profesor/a'})
df_resultados_gen

,Código,Frecuencia,Porcentaje
0,Otros idiomas,71.0,3.59
1,Analizar forma,183.0,9.26
2,Contexto inmediato,279.0,14.11
3,Contexto general,167.0,8.45
4,Elementos no verbales,25.0,1.26
5,Diccionario/Traductor,1042.0,52.71
6,Aplicación IA,61.0,3.09
7,Preguntar compañero/a,65.0,3.29
8,Preguntar profesor/a,84.0,4.25


##### Palabras marcadas como nuevas

In [13]:
df['Num'].describe()

count    140.000000
mean      17.257143
std       10.563526
min        0.000000
25%       10.000000
50%       15.000000
75%       24.000000
max       58.000000
Name: Num, dtype: float64

Aunque algunos participantes no han marcado ninguna palabra en el texto todos han indicado al menos una palabra como "nueva" en la aplicación o en papel (veáse "Número de palabras buscadas").

##### Número de estrategias indicadas (total y por tipo de estrategias)

Estrategias usadas (a elegir, selección múltiple):
- SQ001: Usar mis conocimientos de otros idiomas (lengua materna u otras lenguas extranjeras)
- SQ002: Analizar la forma de la palabra o sus partes
- SQ003: Sacar conclusiones del contexto inmediato (p.ej. relación con otras palabras en la frase)
- SQ004: Sacar conclusiones del contexto general (p.ej. tema del texto o de la clase)
- SQ005: Fijarme en elementos no verbales (p.ej. imágenes, el formato del texto)
- SQ006: Consultar un diccionario o un traductor online
- SQ007: Usar una aplicación de Inteligencia Artificial (p.ej. Chat GPT)
- SQ008: Preguntar a un/a compañero/a
- SQ009: Preguntar al/a la docente

In [14]:
mapeo_columnas = {
    col: col.split("[")[1].split("]")[0] for col in columnas_estrategias
}

df_sumas_individuales = df.groupby(mapeo_columnas, axis=1).sum()

df_sumas_individuales = df_sumas_individuales.add_prefix("Total_")

df_final = pd.concat([df, df_sumas_individuales], axis=1)

columnas_estrategias_totales = [
    col for col in df_final.columns if col.startswith("Total")]

df_final['Total_Estrategias'] = df_final[columnas_estrategias_totales].sum(axis=1)

<ipython-input-14-90bc1e7defda>:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['Total_Estrategias'] = df_final[columnas_estrategias_totales].sum(axis=1)


In [15]:
#df_final[columnas_estrategias_totales].describe().round(2)

##### Porcentaje de tipos de estrategias usadas (en relación con el total de estrategias usadas)

In [16]:
columnas_totales_sq = [col for col in df_final.columns if col.startswith("Total_SQ")]
columnas_porcentajes = {}

for col in columnas_totales_sq:
    codigo_sq = col.replace("Total_", "") 
    nombre_pct = f"Pct_Uso_{codigo_sq}"  

    valores_pct = np.where(
        df_final["Total_Estrategias"] > 0,
        (df_final[col] / df_final["Total_Estrategias"]) * 100,
        0.0,
    )

    columnas_porcentajes[nombre_pct] = np.round(valores_pct, 2)

df_pcts = pd.DataFrame(columnas_porcentajes, index=df_final.index)
df_final = pd.concat([df_final, df_pcts], axis=1)

columnas_estrategias_porcentajes = [
    col for col in df_final.columns if col.startswith("Pct_Uso")]

In [17]:
df_final[columnas_estrategias_porcentajes].describe().round(2)

,Pct_Uso_SQ001,Pct_Uso_SQ002,Pct_Uso_SQ003,Pct_Uso_SQ004,Pct_Uso_SQ005,Pct_Uso_SQ006,Pct_Uso_SQ007,Pct_Uso_SQ008,Pct_Uso_SQ009
count,140.00,140.00,140.00,140.00,140.00,140.00,140.00,140.00,140.00
mean,2.89,7.01,12.82,8.86,1.23,54.51,3.46,3.40,5.82
std,6.07,10.24,12.77,11.92,3.89,28.17,12.09,7.85,14.15
min,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,0.00,0.00,0.00,0.00,0.00,35.61,0.00,0.00,0.00
50%,0.00,0.00,11.11,4.65,0.00,52.40,0.00,0.00,0.00
75%,3.45,12.50,21.63,12.95,0.00,78.43,0.00,0.00,0.86
max,33.33,40.00,50.00,50.00,25.00,100.00,100.00,41.67,85.71


In [18]:
columnas_other = [
    col for col in df_final.columns if col.startswith("E") and "[other]" in col
]
respuestas_texto = []

for col in columnas_other:
    serie_limpia = df_final[col].dropna().astype(str).str.strip()
    serie_valida = serie_limpia[
        ~serie_limpia.isin(["", "0", "nan", "None", "NaN"])
    ]

    respuestas_texto.extend(serie_valida.tolist())

serie_otras_estrategias = pd.Series(respuestas_texto)
conteo_otras_estrategias = serie_otras_estrategias.value_counts()

print("Otras estrategias indicadas por los participantes:")
print(conteo_otras_estrategias)

Otras estrategias indicadas por los participantes:
NO                                                                                    172
diccionario                                                                            25
traductor                                                                               7
Google                                                                                  7
contexto                                                                                6
móvil - Pleco chinese dictionary                                                        4
buscar en el diccionario                                                                4
diccionario/prof                                                                        3
por el contexto                                                                         3
Conocimiento                                                                            3
Libro chino                                      

##### Perfiles de usuarios de estrategias

Análisis de clústeres -> veáse ... (otro notebook) para más detalles

In [19]:
CL = AgglomerativeClustering(n_clusters=6).fit(df_final[columnas_estrategias_porcentajes])
CL.labels_ = CL.labels_+1

In [20]:
df_final["Cluster"] = CL.labels_

<ipython-input-20-87063cdf1c3c>:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final["Cluster"] = CL.labels_


In [21]:
#cambiar por nombres que describen el Cluster?
df_final['Cluster'] = df_final['Cluster'].replace({1:'Combi',
                                                    2:'DiccTrad',
                                                    3:'ProfDicc',
                                                    4:'Contexto',
                                                    5:'IA',
                                                      6:'Prof'})

In [22]:
df_final['Cluster'].value_counts()

Cluster
Combi       52
DiccTrad    48
Contexto    16
ProfDicc    11
IA          10
Prof         3
Name: count, dtype: int64

##### Número de palabras buscadas

In [23]:
columnas_palabras = [col for col in df_final.columns if col.startswith("P") and col[1:].isdigit()]

df_final["Total_Palabras_Buscadas"] = (
    df_final[columnas_palabras]
    .notna()
    .astype(int)
    .sum(axis=1)
    - (df_final[columnas_palabras] == "0").sum(axis=1)
    - (df_final[columnas_palabras] == 0).sum(axis=1)
)

<ipython-input-23-f42a95c63090>:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final["Total_Palabras_Buscadas"] = (


In [24]:
df_final['Total_Palabras_Buscadas'].describe()

count    140.000000
mean      11.392857
std        5.392102
min        1.000000
25%        7.000000
50%       10.000000
75%       16.000000
max       20.000000
Name: Total_Palabras_Buscadas, dtype: float64

##### Ratio estrategias - palabras buscadas

In [25]:
estrategias_por_palabra = np.where(
    df_final["Total_Palabras_Buscadas"] > 0,
    df_final['Total_Estrategias'] / df_final["Total_Palabras_Buscadas"],
    0.0,
)

df_ratio = pd.DataFrame(
    {
        #"Total_Estrategias_Usadas": total_estrategias,
        "Estrategias_Por_Palabra": np.round(estrategias_por_palabra, 2),
    },
    index=df_final.index,
)

df_final = pd.concat([df_final, df_ratio], axis=1)

In [26]:
df_final['Estrategias_Por_Palabra'].describe()

count    140.000000
mean       1.256857
std        0.431197
min        0.290000
25%        1.000000
50%        1.080000
75%        1.400000
max        3.000000
Name: Estrategias_Por_Palabra, dtype: float64

##### Evaluación determinación correcta del significado de las palabras buscadas

- C: Correcto [1 punto]
- O: Otro significado correcto (incorrecto en este contexto) [0 puntos]
- P: Parcialmente correcto [0,5]
- F: Falso [0]
- N: No significado aportado [0]

In [27]:
columnas_totales_ev = [col for col in df_final.columns if col.endswith("_Ev")]

evaluaciones = []

for col in columnas_totales_ev:
    serie_limpia = df_final[col].dropna().astype(str).str.strip().str.replace("С", "C")
    serie_valida = serie_limpia[
        ~serie_limpia.isin(["", "0", "nan", "None", "NaN"])
    ]

    evaluaciones.extend(serie_valida.tolist())

serie_evaluaciones = pd.Series(evaluaciones)
serie_evaluaciones.value_counts()

C    1356
P     106
F      87
O      28
N      19
Name: count, dtype: int64

In [28]:
serie_evaluaciones.value_counts(normalize=True).round(2)

C    0.85
P    0.07
F    0.05
O    0.02
N    0.01
Name: proportion, dtype: float64

In [29]:
df_final['Ev_sum'] = df_final[columnas_totales_ev].replace({"C":1,
                                                "O":0,
                                                "P":0.5,
                                                "F":0,
                                                "N":0}).apply(pd.to_numeric, errors="coerce").sum(axis=1)


<ipython-input-29-9e19b45339e1>:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['Ev_sum'] = df_final[columnas_totales_ev].replace({"C":1,


##### Porcentaje de palabras determinadas correctamente

In [30]:
df_final["Pct_det"] = np.where(
                    df_final["Total_Palabras_Buscadas"] > 0,
                    (df_final["Ev_sum"] / df_final["Total_Palabras_Buscadas"]) * 100,
                    np.nan,  # Se asigna NaN si no buscó ninguna palabra para no distorsionar promedios
                    )

df_final['Pct_det'] = df_final['Pct_det'].round(2)

<ipython-input-30-ae1776337a4f>:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final["Pct_det"] = np.where(


In [31]:
df_final['Pct_det'].describe()

count    140.000000
mean      85.763500
std       16.562918
min        0.000000
25%       81.762500
50%       90.000000
75%       97.137500
max      100.000000
Name: Pct_det, dtype: float64

##### Evaluación del resumen

In [32]:
df_aleman['Resumen'].value_counts()

Resumen
2     26
1     15
3     10
1?     4
Name: count, dtype: int64

In [33]:
df_chino['Resumen'].value_counts()

Resumen
3    32
2    15
1    10
Name: count, dtype: int64

In [34]:
df_ruso['Resumen'].value_counts()

Resumen
2.0    17
3.0     6
1.0     2
Name: count, dtype: int64

In [35]:
df_final['Resumen'] = df_final['Resumen'].replace({"1?":1}).apply(pd.to_numeric, errors="coerce")
df_final['Resumen'].value_counts()

Resumen
2.0    58
3.0    48
1.0    31
0.0     3
Name: count, dtype: int64

#### Guardar datos

In [36]:
df_final.to_pickle('Data_final')

In [37]:
df_final.to_excel('Data_final.xlsx')

In [38]:
#df_final.to_csv('Data_final.csv')